# Overall passing performance
This notebook is designed to import data aggregated by `nomadic summarize`. The notebook can be run one cell at a time (Shift-Enter) or all together ('Run All' above).

In [32]:
from pathlib import Path
import pandas as pd
import matplotlib.ticker as mticker
import sys

from functools import reduce
from typing import List

import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append("../functions")
from upsetplot_fig import upsetplot_fig
from compute_prevalence import compute_variant_prevalence

# Import configuration
from config import (
    summaries_dir,
    output_dir,
    save_results,
    save_format,
    expts_to_exclude,
    workspace_dir,
    results_dir,
    output_config_values_to_user
)


# Settings

In [33]:
output_config_values_to_user()

Configuration determined from your config.yaml file:
Workspace dir         : /home/dan/Work/Projects/NOMADS2/NOMADS2_All_IO2/KEMRI_CDC-NOMADS2_Share/data/workspaces/ATSB_edited
Output dir            : /home/dan/git/resources/notebooks/raw_sequencing_data/results
Summaries dir         : /home/dan/Work/Projects/NOMADS2/NOMADS2_All_IO2/KEMRI_CDC-NOMADS2_Share/data/workspaces/ATSB_edited/summaries/ATSB_edited
Results dir           : /home/dan/Work/Projects/NOMADS2/NOMADS2_All_IO2/KEMRI_CDC-NOMADS2_Share/data/workspaces/ATSB_edited/results
Save results          : False
Save format           : svg
Min prevalence        : None
Experiments excluded  : None
Categories            : []
Barcodes excluded     : None


In [34]:
# Ensure all directories exist
for var in [summaries_dir]:
    if not var.exists():
        print(f"Error: {var} does not exist.")

if save_results:
    output_dir.mkdir(parents=True, exist_ok=True)


# Functions


In [35]:
def rename_amplicons(amp_name):
    if amp_name.startswith("mdr1"):
        return {
            "mdr1-p86-p184": "mdr1-nterm",
            "mdr1-p1034-p1246": "mdr1-cterm",
            "mdr1-p46-245": "mdr1-nterm",
            "mdr1-p968-1278": "mdr1-cterm",
        }[amp_name]
    return amp_name.split("-")[0]

In [36]:
MIN_COVS = [50, 100, 500]  # used further down as well

def load_and_munge_amplicons(
    expt_dir: Path(),
    min_covs: List[int] = MIN_COVS,
) -> pd.DataFrame:
    """
    Load and reformat data pertaining to individual amplicon
    coverage

    TODO:
    - Could add balance statistics
    - Could incorporate coverage relative to negative control

    """

    bedcov_df = pd.read_csv(expt_dir / "summary.region_coverage.csv")
    amp_df = pd.pivot_table(
        index="barcode", columns="name", values="mean_cov", data=bedcov_df
    )
    amp_df.columns = [f"amp_{rename_amplicons(c)}" for c in amp_df.columns]
    AMPLICONS = amp_df.columns

    for min_cov in min_covs:
        amp_df.insert(
            amp_df.shape[1], f"n_amp_gr{min_cov}", (amp_df[AMPLICONS] >= min_cov).sum(1)
        )
    amp_df.reset_index(inplace=True)

    # Mean coverage across all the amplicons
    amp_df.insert(amp_df.shape[1], "amp_mean_cov", amp_df[AMPLICONS].mean(1))

    return amp_df


# Load and concatenate all samples sequenced

In [37]:
sum_coverage = pd.read_csv(summaries_dir / "summary.coverage.csv")
sum_exp_qc = pd.read_csv(summaries_dir / "summary.experiments_qc.csv")

In [ ]:
results = []

workspace_res = workspace_dir / "results"

for expt_dir in results_dir.iterdir():
    
    if not (expt_dir / "metadata").exists():
        print(f"{expt_dir} does not appear to be a valid experiment directory. Skipping....")
        continue
    
    if expts_to_exclude and expt_dir.name in expts_to_exclude:
        print(f"{expt_dir.name} is in the list of experiments to exclude. Skipping....")
        continue

    # Load
    metadata = pd.read_csv(expt_dir / "metadata" / "samples.csv")
    metadata = metadata[["sample_id", "sample_type", "barcode"]]
    reads_df = pd.read_csv(expt_dir / "summary.read_mapping.csv")
    reads_df.drop(columns=["sample_id"], inplace=True)
    amp_df = load_and_munge_amplicons(expt_dir=expt_dir)

    # Merge
    merged_df = reduce(
        lambda x, y: pd.merge(x, y, on="barcode", how="inner"),
        [metadata, reads_df, amp_df],
    )

    merged_df["expt_name"] = expt_dir.name

    # Compute some summaries
    merged_df["per_primary"] = 100 * merged_df["n_primary"] / merged_df["n_total"]
    merged_df["per_mapped"] = 100 * merged_df["n_mapped"] / merged_df["n_total"]

    # Store
    results.append(merged_df)

In [39]:
result_df = pd.concat(results)
result_df = result_df.reset_index(drop=True)
if save_results:
    result_df.to_csv(output_dir / "table.analysis_set.csv", index=False)

# Visualisations

### Panel A: Experiment-level statistics

In [ ]:
stats = ["n_primary", "per_primary", "amp_mean_cov"]
fig, axes = plt.subplots(1, len(stats), figsize=(len(stats) * 3, 5), sharey=True)

fig.subplots_adjust(wspace=0.05)

legend_ax = axes[0]  # use first axis as legend source

for ax, stat in zip(axes, stats):
    sns.stripplot(
        data=result_df, x=stat, y="expt_name", hue="sample_type", dodge=True, ax=ax
    )

    ax.label_outer()
    ax.set_ylabel("")
    ax.set_axisbelow(True) 
    ax.grid(axis="x", which="major", linestyle="-", alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

    # only keep legend on first axis
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# ---- create a single figure-level legend ----
handles, labels = legend_ax.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    # loc="lower center",
    bbox_to_anchor=(1, 1),
    ncol=1,
    frameon=False,
    title="sample_type",
)

if save_results:
    fig.savefig(
        f"{output_dir}/plot.expt.mean_amp_cov.{save_format}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.5,
    )

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

sns.stripplot(
    y="expt_name",
    x="amp_mean_cov",
    hue="sample_type",
    dodge=True,
    data=result_df,
    ax=ax
)

ax.set_axisbelow(True)
ax.grid(ls="dotted", which="minor", axis="x", alpha=0.5)
ax.grid(ls="solid", which="major", axis="x")

ax.set_ylabel("")
ax.set_xscale("log")
ax.set_xlabel("Mean Amplicon Coverage")
ax.legend(bbox_to_anchor=(1, 1))

if save_results:
    fig.savefig(
        f"{output_dir}/plot.expt.mean_amp_cov.svg",
        bbox_inches="tight",
        # pad_inches=0.5,
    )


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=result_df,
    x="amp_mean_cov",
    y="sample_type",
    hue="sample_type",
    ax=ax,
)

ax.set_axisbelow(True)
ax.grid(axis="x", which="major", linestyle="-", alpha=0.3)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.set_xlabel("Mean Amplicon Coverage")
ax.set_ylabel("")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=result_df,
    x="amp_mean_cov",
    y="expt_name",
    hue="sample_type",
    ax=ax,
)

ax.set_axisbelow(True)
ax.grid(axis="x", which="major", linestyle="-", alpha=0.3)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.set_xlabel("Mean Amplicon Coverage")
ax.set_ylabel("")

### Panel B: Per-amplicon pass rates

In [47]:
primary_df = result_df.copy(deep=True)

AMPLICONS = [
    c for c in primary_df.columns if c.startswith("amp_") and not c.endswith("_cov")
]
IDS = ["sample_id"]

In [48]:
tall_df = pd.melt(
    frame=primary_df[IDS + AMPLICONS],
    id_vars=IDS,
    value_vars=AMPLICONS,
    var_name="amplicon",
    value_name="mean_cov",
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4))

sns.stripplot(
    x="mean_cov",
    y="amplicon",
    hue="amplicon",
    s=4,
    alpha=0.8,
    legend=False,
    data=tall_df,
)

sns.boxplot(
    x="mean_cov",
    y="amplicon",
    data=tall_df,
    showcaps=False,
    showfliers=False,
    boxprops={"facecolor": "None"},
    whiskerprops={"linewidth": 0},
    ax=ax,
)

ax.set_xscale("log")
ax.set_axisbelow(True)
ax.grid(ls="dotted", axis="x")

label_median = True
if label_median:
    for i, (amplicon, amp_df) in enumerate(tall_df.groupby("amplicon")):
        med = amp_df["mean_cov"].median()
        ax.annotate(
            xy=(ax.get_xlim()[1], i), ha="left", va="center", text=f" {int(med)}"
        )

ax.set_xlabel("Mean Coverage")
ax.set_ylabel("")

if save_results:
    fig.savefig(
        f"{output_dir}/plot.amp.median_mean_cov.{save_format}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.5,
    )


### Panel C: Pass rate per amplicon

In [50]:
results = []
for min_cov in MIN_COVS:
    results.append((primary_df[AMPLICONS] > min_cov).sum())
# these would be initial numbers
thres_df = pd.concat(results, axis=1)
thres_df.columns = [f"$>${m}$\\times$" for m in MIN_COVS]

percent_thres_df = 100 * thres_df / primary_df.shape[0]

In [ ]:
fig, ax = plt.subplots(1, 1)
thres_df.plot(kind="bar", ax=ax)


ax.set_ylim(0, primary_df.shape[0])
ax.yaxis.set_major_locator(plt.MaxNLocator(10))
# ax.yaxis.set_major_locator(plt.MultipleLocator(10))
ax.grid(ls="dotted")
ax.set_ylabel("No. Samples")


axm = ax.twinx()
axm.set_ylim((0, 100))
axm.yaxis.set_major_locator(plt.MaxNLocator(10))
# axm.yaxis.set_major_locator(plt.MultipleLocator(10))
axm.set_ylabel("(%)")

ax.legend(bbox_to_anchor=(1.1, 1), ncol=1)

if save_results:
    fig.savefig(
        f"{output_dir}/plot.amp.cov_threshold.{save_format}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.5,
    )

In [52]:
cols = {
    "n_amp_gr50": "$50\\times$",
    "n_amp_gr100": "$100\\times$",
    "n_amp_gr500": "$500\\times$",
}
outputs = []
for c, cname in cols.items():
    for v in range(11):
        outputs.append((cname, v, sum(primary_df[c] >= v)))

In [53]:
count_df = pd.DataFrame(outputs)
count_df.columns = ["threshold", "min_amps", "n_samples"]

In [54]:
count_df["per_samples"] = 100 * count_df["n_samples"] / primary_df.shape[0]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.set_axisbelow(True)
for _, t in cols.items():  # have to do this to match colors
    tdf = count_df.query("threshold == @t")
    ax.plot(
        tdf["min_amps"], tdf["per_samples"], marker="o", ms=4, label=t, clip_on=False
    )
ax.legend(loc="lower left")

ax.yaxis.set_major_locator(plt.MultipleLocator(10))
ax.yaxis.set_minor_locator(plt.MultipleLocator(5))

ax.xaxis.set_major_locator(plt.MultipleLocator(1))


ax.grid(ls="dotted", axis="y", which="minor", alpha=0.5)
ax.grid(ls="solid", axis="y", which="major", alpha=0.5)
# ax.grid(ls='solid', axis='x', which='major', alpha=0.5)


ax.set_ylim((0, 100))
ax.set_xlim((0, 10))


ax.axvline(8, lw=2, color="darkgrey", alpha=0.5)

ax.set_ylabel("Percent of Samples (%)")
ax.set_xlabel("No. Amplicons $>=$ Threshold")

if save_results:
    fig.savefig(
        f"{output_dir}/plot.amp.cov_trend.{save_format}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.5,
    )

### Panel D. Understanding sample characteristics that correlate with coverage

In [96]:
# Pull in metadata from master data file
master_df = pd.read_csv(summaries_dir / "metadata.csv")
# Merge define cols with primary_df
master_cols = ["sample_id", "lab_asexual_parasite_thick", "lab_asexual_parasite_thin", "enrl_region", "enrl_sex"]
secondary_df = primary_df.merge(master_df[master_cols], on="sample_id", how="left")

In [ ]:
for var in master_cols[1:]:
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))

    if pd.api.types.is_numeric_dtype(secondary_df[var]):
        # ---- numeric variable ----
        sns.scatterplot(
            x=var,
            y="amp_mean_cov",
            data=secondary_df,
            ax=ax,
            alpha=0.8,
        )
        ax.set_xscale("log")

    else:
        # ---- categorical variable ----
        sns.stripplot(
            x=var,
            y="amp_mean_cov",
            hue=var,
            data=secondary_df,
            ax=ax,
            dodge=True,
            alpha=0.8,
        )
        ax.legend(bbox_to_anchor=(1, 1))

    ax.set_yscale("log")
    ax.set_axisbelow(True)
    ax.grid(ls="dotted", axis="y", which="minor", alpha=0.5)
    ax.grid(ls="solid", axis="y", which="major", alpha=0.5)

    ax.set_xlabel(var)
    ax.set_ylabel("Mean Amplicon Coverage")
